# 04. Censored Clinical Trial Biostatistics — `timeROC`, `survIDINRI`, `survRM2`, `BuyseTest` & `JMbayes2`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gillanilab/pythonic_biotools/blob/main/notebooks/04_clinical_trials_winratio_rmst_and_joint_survival.ipynb)

This tutorial walks through a complete Phase III oncology clinical trial & biomarker evaluation workflow across five specialized right-censored biostatistics libraries in `pythonic_biotools`:

1. **`timeroc`** — Inverse-Probability-of-Censoring-Weighted (`IPCW`) time-dependent ROC curves, horizon AUCs, asymptotic IID standard errors, and paired biomarker comparisons.
2. **`survidinri`** — Censored Integrated Discrimination Improvement (`IDI`) and continuous Net Reclassification Improvement (`cNRI`, highlighting Uno's `R_default` half-scale vs. full-scale convention).
3. **`survrm2`** — Unadjusted and ANCOVA covariate-adjusted Restricted Mean Survival Time (`RMST` & `RMTL`) differences and ratios under non-proportional hazards.
4. **`buysetest`** — Stratified Generalized Pairwise Comparisons (`Win Ratio`, `Net Benefit`, `Win Odds` via Péron KM scoring) and covariate-adjusted Proportional Win-Fractions Regression (`pwreg`).
5. **`jmbayes2`** — Bayesian joint longitudinal biomarker + survival modeling (15-node Gauss–Kronrod quadrature + Adaptive MCMC) and individualized dynamic landmark risk prediction.

In [1]:
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/gillanilab/pythonic_biotools.git"]
    )
else:
    repo_root = Path.cwd().resolve()
    if not (repo_root / "pythonic_biotools").is_dir() and (repo_root.parent / "pythonic_biotools").is_dir():
        repo_root = repo_root.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

from pythonic_biotools import buysetest, jmbayes2, survidinri, survrm2, timeroc

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 15)
print("Loaded clinical trial survival toolkit: timeroc, survidinri, survrm2, buysetest, jmbayes2")

Loaded clinical trial survival toolkit: timeroc, survidinri, survrm2, buysetest, jmbayes2


## 1. Simulate a Randomized Phase III Trial Cohort ($N = 160$)

Patients are randomized 1:1 to Control (`arm = 0`) vs. Experimental Immunotherapy (`arm = 1`), with baseline clinical `ecog` status, a baseline `ctdna_score` biomarker, progression-free survival (`pfs_time`), overall survival (`os_time`), and grade 3+ toxicity.

In [2]:
rng = np.random.default_rng(2026)
n = 160
arm = np.repeat([0, 1], n // 2)
ecog = rng.binomial(1, 0.45, size=n).astype(float)
ctdna_score = rng.normal(0.0, 1.0, size=n)
clinical_only_marker = 0.45 * ecog + rng.normal(0.0, 0.6, size=n)

log_hazard = -0.65 * arm + 0.85 * ctdna_score + 0.45 * ecog
t_os_true = rng.exponential(scale=np.exp(-log_hazard) * 7.5)
t_pfs_true = np.minimum(t_os_true, rng.exponential(scale=np.exp(-log_hazard) * 4.5))
t_cens = rng.uniform(5.0, 16.0, size=n)

os_time = np.minimum(t_os_true, t_cens)
os_event = (t_os_true <= t_cens).astype(int)
pfs_time = np.minimum(t_pfs_true, t_cens)
pfs_event = (t_pfs_true <= t_cens).astype(int)
grade3_tox = rng.binomial(1, np.where(arm == 1, 0.14, 0.34))

trial_df = pd.DataFrame(
    {
        "id": np.arange(n),
        "arm": arm,
        "ecog": ecog,
        "ctdna_score": ctdna_score,
        "pfs_time": pfs_time,
        "pfs_event": pfs_event,
        "os_time": os_time,
        "os_event": os_event,
        "grade3_tox": grade3_tox,
    }
)
print(f"Simulated N={n} patients | OS events: {os_event.sum()} ({os_event.mean():.1%}) | PFS events: {pfs_event.sum()}")
print(trial_df.head(5).round(3).to_string(index=False))

Simulated N=160 patients | OS events: 105 (65.6%) | PFS events: 140
 id  arm  ecog  ctdna_score  pfs_time  pfs_event  os_time  os_event  grade3_tox
  0    0   0.0       -0.852     4.940          1    4.940         1           0
  1    0   1.0        0.849     0.341          1    0.952         1           1
  2    0   0.0       -1.426     5.080          0    5.080         0           0
  3    0   0.0       -0.453     4.827          1   12.131         0           0
  4    0   0.0       -2.253    11.706          0   11.706         0           0


## 2. Time-Dependent Prognostic Biomarker Discrimination (`timeroc` & `survidinri`)

We evaluate `ctdna_score` vs. `clinical_only_marker` at $t = 3, 5, 8$ months via IPCW `time_roc`, test paired AUC differences with `compare_timeroc`, and quantify incremental risk reclassification (`IDI` and `cNRI`) when adding `ctdna_score` to `ecog` at $t_0 = 6.0$ months.

In [3]:
horizons = [3.0, 5.0, 8.0]
roc_ctdna = timeroc.time_roc(
    T=os_time, delta=os_event, marker=ctdna_score, times=horizons, cause=1, iid=True, settings="default"
)
roc_clin = timeroc.time_roc(
    T=os_time, delta=os_event, marker=clinical_only_marker, times=horizons, cause=1, iid=True, settings="default"
)
comp_roc = timeroc.compare_timeroc(roc_ctdna, roc_clin, seed=42)

roc_tbl = pd.DataFrame(
    {
        "horizon_mo": horizons,
        "AUC_ctDNA": roc_ctdna.auc_1,
        "SE_ctDNA": roc_ctdna.se_auc_1,
        "AUC_clinical": roc_clin.auc_1,
        "p_diff": comp_roc.p_values,
    }
)
print("Time-dependent ROC comparison:")
print(roc_tbl.round(4).to_string(index=False))

cov0 = trial_df[["ecog"]].to_numpy()
cov1 = trial_df[["ecog", "ctdna_score"]].to_numpy()
idi_def = survidinri.idi_inf(os_time, os_event, cov0, cov1, t0=6.0, npert=120, seed=42, settings="default")
idi_r = survidinri.idi_inf(os_time, os_event, cov0, cov1, t0=6.0, npert=120, seed=42, settings="R_default")

print(
    f"\n[survIDINRI @ t0=6.0] IDI = {idi_def.m1_idi.est:.4f} (p={idi_def.m1_idi.pval:.3e}) | "
    f"cNRI (default full [-2,2] scale) = {idi_def.m2_nri.est:.4f} | "
    f"cNRI (R_default Uno half [-1,1] scale) = {idi_r.m2_nri.est:.4f}"
)

Time-dependent ROC comparison:
 horizon_mo  AUC_ctDNA  SE_ctDNA  AUC_clinical  p_diff
        3.0     0.8104    0.0361        0.5005     0.0
        5.0     0.8296    0.0319        0.5181     0.0
        8.0     0.8007    0.0376        0.4833     0.0

[survIDINRI @ t0=6.0] IDI = 0.2682 (p=4.482e-09) | cNRI (default full [-2,2] scale) = 1.0327 | cNRI (R_default Uno half [-1,1] scale) = 0.5163


## 3. Restricted Mean Survival Time (`survrm2`) & Hierarchical Composite Win Ratio (`buysetest`)

We estimate unadjusted and covariate-adjusted 10-month `RMST` gain (`rmst2`), followed by a 3-layer prioritized composite endpoint (`OS` $\succ$ `PFS` $\succ$ absence of `grade3_tox`) using Péron Kaplan–Meier pairwise scoring (`buyse_test`) and covariate-adjusted Proportional Win-Fractions Regression (`pwreg`) across prioritized TTE layers.

In [4]:
rmst_res = survrm2.rmst2(os_time, os_event, arm, tau=10.0, covariates=cov1, settings="default")
print(
    f"[survRM2 @ tau=10.0] Unadjusted RMST Arm1={rmst_res.arm1.rmst:.3f}m vs Arm0={rmst_res.arm0.rmst:.3f}m | "
    f"Diff={rmst_res.rmst_diff_10.est:.3f}m (95% CI [{rmst_res.rmst_diff_10.ci_lower:.3f}, {rmst_res.rmst_diff_10.ci_upper:.3f}], "
    f"p={rmst_res.rmst_diff_10.pval:.3e})"
)
print(
    f"[survRM2 ANCOVA] Covariate-adjusted RMST Diff={rmst_res.adj_rmst_diff_10.est:.3f}m "
    f"(p={rmst_res.adj_rmst_diff_10.pval:.3e})"
)

endpoints = [
    buysetest.Endpoint("os_time", kind="tte", status_col="os_event", operator="higher", threshold=0.5),
    buysetest.Endpoint("pfs_time", kind="tte", status_col="pfs_event", operator="higher", threshold=0.25),
    buysetest.Endpoint("grade3_tox", kind="binary", operator="lower"),
]
bt_def = buysetest.buyse_test(trial_df, treatment="arm", endpoints=endpoints, settings="default")
bt_r = buysetest.buyse_test(trial_df, treatment="arm", endpoints=endpoints, settings="R_default")

print("\n[BuyseTest] Hierarchical Generalized Pairwise Comparisons (Péron scoring):")
print(
    bt_def.to_dataframe()[["endpoint", "favorable", "unfavorable", "net_benefit", "win_ratio", "win_ratio_pval"]]
    .round(4)
    .to_string(index=False)
)
print(
    f"2nd-order H-decomposition SE parity check (Net Benefit SE): "
    f"default={bt_def.net_benefit.se[-1]:.5f} vs R_default={bt_r.net_benefit.se[-1]:.5f}"
)

fit_pw = buysetest.pwreg(
    trial_df,
    id_col="id",
    treatment_col="arm",
    covariates=["ecog", "ctdna_score"],
    endpoints=endpoints[:2],
    settings="default",
)
pw_tbl = pd.DataFrame(
    {
        "term": fit_pw.coef_names,
        "log_WR": fit_pw.coef,
        "win_ratio": fit_pw.win_ratio,
        "CI_low": fit_pw.ci_lower,
        "CI_high": fit_pw.ci_upper,
        "pval": fit_pw.pval,
    }
)
print("\n[WR::pwreg] Covariate-adjusted Proportional Win-Fractions Regression (OS > PFS):")
print(pw_tbl.round(4).to_string(index=False))

[survRM2 @ tau=10.0] Unadjusted RMST Arm1=6.409m vs Arm0=5.868m | Diff=0.541m (95% CI [-0.596, 1.677], p=3.511e-01)
[survRM2 ANCOVA] Covariate-adjusted RMST Diff=0.913m (p=7.957e-02)

[BuyseTest] Hierarchical Generalized Pairwise Comparisons (Péron scoring):
  endpoint  favorable  unfavorable  net_benefit  win_ratio  win_ratio_pval
   os_time     0.5044       0.3964       0.1080     1.2725          0.2221
  pfs_time     0.5605       0.4262       0.1342     1.3149          0.1343
grade3_tox     0.5661       0.4270       0.1391     1.3257          0.1208
2nd-order H-decomposition SE parity check (Net Benefit SE): default=0.08848 vs R_default=0.08870

[WR::pwreg] Covariate-adjusted Proportional Win-Fractions Regression (OS > PFS):
       term  log_WR  win_ratio  CI_low  CI_high   pval
        arm  0.6684     1.9511  1.2948   2.9402 0.0014
       ecog -0.4065     0.6660  0.4464   0.9935 0.0464
ctdna_score -1.1153     0.3278  0.2569   0.4183 0.0000


## 4. Bayesian Joint Longitudinal Biomarker + Survival Modeling (`jmbayes2`)

In precision oncology trials, serial on-treatment `ctDNA` measurements dynamically track disease progression. We fit a Bayesian joint longitudinal-survival model (`jm`) and generate dynamic landmark survival predictions (`predict_dynamic`) for a patient monitored through month `2.0`.

In [5]:
n_jm = 55
surv_jm = trial_df.iloc[:n_jm][["id", "os_time", "os_event", "arm"]].rename(
    columns={"os_time": "time", "os_event": "event"}
)
long_rows = []
for row in surv_jm.itertuples(index=False):
    visit_times = [t for t in (0.0, 1.0, 2.0, 3.5, 5.0) if t < row.time] or [0.0]
    subj_intercept = ctdna_score[int(row.id)]
    for tj in visit_times:
        slope_arm = -0.18 if row.arm == 1 else 0.22
        long_rows.append(
            {
                "id": int(row.id),
                "obstime": tj,
                "y": subj_intercept + slope_arm * tj + rng.normal(0.0, 0.20),
                "arm": float(row.arm),
            }
        )
long_jm = pd.DataFrame(long_rows)

jm_fit = jmbayes2.jm(
    surv_jm,
    long_jm,
    surv_covs=["arm"],
    long_covs=["obstime", "arm"],
    n_iter=240,
    burn_in=80,
    seed=42,
    settings="default",
)
print("JMbayes2 Posterior Association Summary (current biomarker value -> log-hazard):")
print(jm_fit.assoc_df.round(4).to_string(index=False))

patient_serial = pd.DataFrame(
    {
        "id": [999, 999, 999],
        "obstime": [0.0, 1.0, 2.0],
        "y": [0.40, 0.75, 1.15],
        "arm": [0.0, 0.0, 0.0],
    }
)
dyn_pred = jmbayes2.predict_dynamic(
    jm_fit,
    newdata_long_df=patient_serial,
    t_landmark=2.0,
    horizons=[4.0, 6.0, 8.0],
    surv_covs_row={"arm": 0.0},
    n_mcmc=100,
    seed=42,
    settings="default",
)
print("\nDynamic post-landmark survival & event probabilities (landmark t = 2.0 months):")
print(dyn_pred.round(4).to_string(index=False))

JMbayes2 Posterior Association Summary (current biomarker value -> log-hazard):
 cause  form   mean     sd  median  ci_lower  ci_upper     hr  hr_ci_lower  hr_ci_upper   pval   rhat
     1 value 0.5811 0.1341  0.5537    0.3764    0.8316 1.8043        1.457        2.297 0.0062 1.0235

Dynamic post-landmark survival & event probabilities (landmark t = 2.0 months):
 id  t_landmark  horizon  n_obs_used  surv_prob  surv_sd  surv_ci_lower  surv_ci_upper
999         2.0      4.0           3     0.7155   0.0252         0.6717         0.7621
999         2.0      6.0           3     0.5065   0.0451         0.4321         0.5818
999         2.0      8.0           3     0.3453   0.0594         0.2538         0.4392
